In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
import pickle

In [2]:
df = pd.read_csv("./data/Churn_Modelling.csv")

In [3]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


## Preprocessing


In [4]:
df = df.drop(["RowNumber", "CustomerId", "Surname", "Exited"], axis=1)

# Encode categorical variables
gender_label_encoder = LabelEncoder()

df["Gender"] = gender_label_encoder.fit_transform(df["Gender"])

one_hot_geography_encoder = OneHotEncoder(
    drop="first"
)  # drop='first' to avoid dummy variable trap

df = pd.concat(
    [
        df.drop("Geography", axis=1),
        pd.DataFrame(
            one_hot_geography_encoder.fit_transform(df[["Geography"]]).toarray(),
            columns=one_hot_geography_encoder.get_feature_names_out(["Geography"]),
        ),
    ],
    axis=1,
)

X = df.drop("EstimatedSalary", axis=1)
y = df["EstimatedSalary"]

In [5]:
X.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,0.0,1.0


In [6]:
y

0       101348.88
1       112542.58
2       113931.57
3        93826.63
4        79084.10
          ...    
9995     96270.64
9996    101699.77
9997     42085.58
9998     92888.52
9999     38190.78
Name: EstimatedSalary, Length: 10000, dtype: float64

## Train Test split and Scaling


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## Saving the preprocessing objects


In [8]:
with open("./models/reg_gender_label_encoder.pkl", "wb") as f:
    pickle.dump(gender_label_encoder, f)

with open("./models/reg_one_hot_geography_encoder.pkl", "wb") as f:
    pickle.dump(one_hot_geography_encoder, f)

with open("./models/reg_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

## ANN Reg Implementation


In [9]:
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

In [10]:
model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(X_train.shape[1],)),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(1, activation="linear"),
    ]
)

model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"],
)

In [11]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,089 (35.50 KB)

 Trainable params: 9,089 (35.50 KB)

 Non-trainable params: 0 (0.00 B)

In [14]:
LOG_DIR = f"logs/fit/{datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}"

early_stopping_callback = EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)

model.fit(
    X_train,
    y_train,
    epochs=100,
    validation_data=(X_test, y_test),
    callbacks=[
        early_stopping_callback,
        TensorBoard(log_dir=LOG_DIR),
    ],
)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 3308376320.0000 - mae: 49550.1914 - val_loss: 3363046144.0000 - val_mae: 50071.7617
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 3305498624.0000 - mae: 49536.6094 - val_loss: 3362163200.0000 - val_mae: 50083.4766
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 3303557120.0000 - mae: 49504.4922 - val_loss: 3368859392.0000 - val_mae: 50145.6953
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 3302213888.0000 - mae: 49500.0547 - val_loss: 3361464576.0000 - val_mae: 50068.2383
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 3301241600.0000 - mae: 49501.0586 - val_loss: 3362747648.0000 - val_mae: 50090.3164
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 3297244160.0000 - mae: 49460.8945 - val_loss: 3375834368.0000 - val_mae: 50191.2422
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 3297297664.0000 - mae: 49485.5352 - val_loss: 3363027712.0000 - val_m

In [15]:
model.save("./models/reg_churn_model.h5")

## Evaluation


In [17]:
from sklearn.metrics import r2_score

test_loss, test_mae = model.evaluate(X_test, y_test)
print(f"Test Loss: {test_loss}, Test MAE: {test_mae}")

y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
print(f"Test R2 Score: {r2}")

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 3361464576.0000 - mae: 50068.2383  
Test Loss: 3361464576.0, Test MAE: 50068.23828125
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
Test R2 Score: -0.01823896301714867
